# 第8回 単位根とランダムウォーク


第7回では、月ごとの変動の大きさ（その月の日々の対数変化率の標準偏差）に AR(1) モデル $y_t = c + \phi\, y_{t-1} + \varepsilon_t$ を当てはめた。傾き $\phi$ は0.507で、来月の予想は前月の値と過去の平均の中間にきた。

この回では、東京17時のドル・円レートそのもの（1ドルが何円か）を考える。レートの値に同じ式を当てはめると、$\phi$ はほぼ1になる。$\phi$ が1のとき系列はどう動き、過去の平均は予想の役に立つのだろうか。


## 実行の準備

`finance_data.zip` をNotebookと同じフォルダに置き、下の2つのセルを実行する。2つ目のセルは `statsmodels` を読み込む。


In [ ]:
from pathlib import Path
from zipfile import ZipFile

if not Path("course_data.py").exists():
    with ZipFile("finance_data.zip") as archive:
        archive.extractall(".")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from course_data import (
    load_boj_fx, load_boj_monthly, load_fed_fx, load_fed_rates,
    configure_plotting, format_axis,
)

configure_plotting()
pd.options.display.float_format = "{:.3f}".format


In [ ]:
import sys
if sys.platform == "emscripten":
    import pyodide_js
    await pyodide_js.loadPackage("statsmodels")
import statsmodels.api as sm


## 第7回 演習1の解説

第7回の演習1のすべての問いを振り返ろう。まず第7回と同じ手順で、月ごとの変動の大きさ `vol` を作り、2000〜2019年の値で AR(1) モデルを当てはめ直す。切片を `c_vol`、傾きを `phi_vol`、2000〜2019年の平均を `train_mean` とする。


In [ ]:
boj = load_boj_fx()
rate = boj["usd_jpy"].dropna()
log_pct = 100 * np.log(rate).diff()
vol = log_pct.resample("ME").std()
vols = pd.DataFrame({
    "前月の値": vol.shift(1),
    "今月の値": vol,
}).dropna()

train_vol = vols.loc[:"2019"]
model_vol = sm.OLS(train_vol["今月の値"], sm.add_constant(train_vol["前月の値"])).fit()
c_vol = model_vol.params["const"]
phi_vol = model_vol.params["前月の値"]
train_mean = vol.loc[:"2019"].mean()
print(f"切片 c：{c_vol:.3f}、傾き φ：{phi_vol:.3f}、2000〜2019年の平均：{train_mean:.3f}%")


### 問1：ユーロ・ドルの傾き

問1は、FRBのニューヨーク正午のユーロ・ドルについて、同じ手順で月ごとの変動の大きさに AR(1) モデルを当てはめ、傾きをドル・円の0.507と比べる問いだった。


In [ ]:
fed = load_fed_fx()
eur_log_pct = 100 * np.log(fed["eur_usd"].dropna()).diff()
eur_vol = eur_log_pct.resample("ME").std()
eur_vols = pd.DataFrame({
    "前月の値": eur_vol.shift(1),
    "今月の値": eur_vol,
}).dropna()
eur_train = eur_vols.loc[:"2019"]
model_eur = sm.OLS(eur_train["今月の値"], sm.add_constant(eur_train["前月の値"])).fit()
print(f"ユーロ・ドルの傾き φ：{model_eur.params['前月の値']:.3f}")


ユーロ・ドルの傾きは0.705で、ドル・円の0.507より大きい。傾きが0より大きいので、ユーロ・ドルでも変動の大きい月の次の月は変動が大きくなりやすいといえる。傾きが1に近いほど、今月の予想は前月の値に近くなる。


### 問2：3つの文の正誤

(a) 「前日の変化に当てはめた AR(1) モデルの傾きは −0.0142 と負なので、前日に円安だった次の日は円高になると予想でき、今日の値による予想より誤差がずっと小さくなる。」

誤りである。傾きが負なので、前日に円安へ動いた次の日の予想は、たしかに円高の向きになる。問題は、その予想の大きさである。第7回と同じく2000〜2023年の値で前日の変化に AR(1) モデルを当てはめ、前日の変化がいくつのときに当日の変化の予想がどうなるかを計算しよう。


In [ ]:
change = rate.diff()
changes = pd.DataFrame({
    "前日の変化": change.shift(1),
    "当日の変化": change,
}).dropna()
train_change = changes.loc[:"2023"]
model_change = sm.OLS(train_change["当日の変化"], sm.add_constant(train_change["前日の変化"])).fit()
c_change = model_change.params["const"]
phi_change = model_change.params["前日の変化"]
for previous in [-5, -2, 0, 2, 5]:
    print(f"前日の変化 {previous:+d} 円 → 当日の変化の予想 {c_change + phi_change * previous:+.3f} 円")


前日に5円の円安という大きな動きがあっても、当日の変化の予想は −0.064 円の円高にとどまる。第5回で求めたように、2024年の1日の変化の大きさの平均は0.70円ほどあり、0.06円の予想はその10分の1にも満たない。予想が円高の向きでも、実際の動きに比べてごく小さいので、予想の誤差はほとんど変わらない。第7回で求めた2024年の誤差の大きさの平均も、AR(1) モデルで0.694円、今日のレートをそのまま使う予想で0.696円と、ほぼ同じだった。傾きは符号だけでなく、予想がどのくらい動くかという大きさも合わせて読む。

(b) 「AR(1) モデルの傾き φ が0なら、予想は前の値によらず、いつも切片 c と同じ値になる。」

正しい。予想は $c + \phi\, y_{t-1}$ で、$\phi = 0$ なら $\phi\, y_{t-1}$ は $y_{t-1}$ がどんな値でも0になり、予想は $c$ だけが残る。

(c) 「月ごとの変動の大きさの AR(1) モデル（傾き0.507）では、前月の変動の大きさが学習期間の平均より高いとき、今月の予想は前月の値より高くなる。」

誤りである。前月が1.000%のときの予想を計算してみよう。


In [ ]:
previous = 1.0
forecast = c_vol + phi_vol * previous
print(f"前月 {previous:.3f}% → 今月の予想 {forecast:.3f}%（平均 {train_mean:.3f}%）")
print(f"前月と平均の差：{previous - train_mean:.3f}%、予想と平均の差：{forecast - train_mean:.3f}%")


前月が1.000%のとき、今月の予想は0.795%で、前月の値より低い。

予想がどこにくるかは、予想の式が直線であることから分かる。第7回で見たように、前月が平均の0.585%なら、予想もほぼ平均になる。直線の傾きは0.507なので、前月の値が平均から0.415%離れると、予想は平均から 0.507 × 0.415 = 0.210% 離れる。表示された差もそのとおり0.210%で、前月と平均の差のおよそ半分である。

傾きが0から1の間にある AR(1) モデルでは、予想と平均の差は、前月と平均の差の $\phi$ 倍になる。予想はいつも前月の値と平均の間にきて、前月の値より平均に近い。「前月が平均より高ければ、今月の予想も平均より高い」ことは正しいが、「前月の値より高い」ことにはならない。


### 問3：AさんとBさん

問3は、明日1000ドルを両替するAさん（明日のレートが上がるか下がるかを知りたい）と、来月の旅行の予算を立てるBさん（来月の日々のレートがどのくらい大きく動きそうかを知りたい）にとって、AR(1) モデルの結果が役に立つかを説明する問いだった。

まず、2人の知りたいことが本文のどの AR(1) モデルにあたるかを分けて考える。

- Aさんの知りたいこと（明日の向き）には、前日の変化に当てはめた AR(1) モデルが対応する。
- Bさんの知りたいこと（来月の変動の大きさ）には、月ごとの変動の大きさの AR(1) モデルが対応する。

解答には、2人それぞれについて、対応するモデル、傾きの値、誤差の大きさの平均の比較の3つが入っているとよい。Bさんについては、予想にも誤差があることも添えたい。2024年8月は、予想の0.717%に対して実際は1.177%で、予想より大きく動いた。

解答の例は次のとおり。

> Aさんにとって、AR(1) モデルはほとんど役に立たない。前日の変化に当てはめた傾きは −0.0142 とほぼ0で、2024年の誤差の大きさの平均も0.694円と、今日のレートをそのまま使う予想の0.696円とほぼ同じだったからである。Bさんにとっては役に立つ。月ごとの変動の大きさの傾きは0.507で、2020〜2025年の1か月先の予想の誤差の大きさの平均0.161%は、前月の値（0.177%）や2000〜2019年の平均（0.195%）による予想より小さかった。ただし2024年8月のように予想より大きく動く月もあるので、予算には余裕を持たせたい。


### 追加問題：2026年1月と2月の予想

追加問題は、2025年12月の変動の大きさから、2026年1月と2月の変動の大きさを予想する問いだった。1月の予想は2025年12月の値を式に当てはめて求め、2月の予想は1月の予想を前月の値として式に当てはめて求める。同じ計算を6か月先まで続け、各月の予想と平均の差も表示しよう。


In [ ]:
forecast = vol.loc["2025-12-31"]
print(f"2025年12月の値：{forecast:.3f}%（平均との差 {forecast - train_mean:+.3f}%）")
for month in range(1, 7):
    forecast = c_vol + phi_vol * forecast
    print(f"2026年{month}月の予想：{forecast:.3f}%（平均との差 {forecast - train_mean:+.3f}%）")
print(f"2000〜2019年の平均：{train_mean:.3f}%")


1月の予想は0.488%、2月は0.535%で、先の月ほど2000〜2019年の平均0.585%に近づき、6月には0.580%になった。

平均との差を見ると、2025年12月の −0.190% が、1月に −0.097%、2月に −0.050%、3月に −0.026% と、1か月ごとにほぼ半分になっている。(c) で見たとおり、予想と平均の差は1か月ごとに φ = 0.507 倍になるからである。2か月先の差は 0.507 × 0.507 = 0.257 倍、3か月先は0.13倍で、k か月先の差は12月の差の φ の k 乗倍になる。予想は2025年12月の0.395%から離れ、平均へ戻っていく。

φ = 1 なら、1の k 乗はいつまでも1なので、差はまったく縮まない。この回では、φ が1に近い系列を調べる。


## φ が1に近い系列

第7回と同じく、$c = 0$、$\varepsilon_t$ を平均0、標準偏差1の乱数として、AR(1) モデルの式で系列を作る。今度は $\phi = 0.5$、$0.9$、$1.0$ の3つを、同じ $\varepsilon_t$ で200期分作り、1つの図に重ねる。


In [ ]:
rng = np.random.default_rng(7)
eps = rng.normal(size=200)

fig, ax = plt.subplots(figsize=(9, 4))
for phi in [0.5, 0.9, 1.0]:
    y = np.zeros(200)
    for t in range(1, 200):
        y[t] = phi * y[t - 1] + eps[t]
    ax.plot(y, linewidth=0.8, label=f"φ = {phi}")
ax.axhline(0, color="gray", linewidth=0.8)
ax.set(title="φ が1に近い AR(1) モデルの系列", xlabel="t", ylabel="y")
ax.legend()
format_axis(ax, dates=False)
fig.tight_layout()
plt.show()


φ = 0.5 の系列（青）は0のまわりにとどまる。φ = 0.9 の系列（オレンジ）は0から離れる時期が長くなるが、やがて0へ戻る。φ = 1.0 の系列（緑）は、t が20のあたりで0から離れたあと戻らず、最後の t = 199 では −26 付近にある。3つの系列は同じ $\varepsilon_t$ から作ったので、この違いは $\phi$ だけから生じている。

$\phi = 1$、$c = 0$ の AR(1) モデル

$$y_t = y_{t-1} + \varepsilon_t$$

をドリフトのないランダムウォークという。今の値は1期前の値に $\varepsilon_t$ を足しただけで、0へ戻す力が働かない。

$\phi = 1$ で $c$ が0でないモデルは $y_t = c + y_{t-1} + \varepsilon_t$ で、今の値は1期前の値に $c$ と $\varepsilon_t$ を足した値になる。$\varepsilon_t$ の平均は0なので、系列は毎期平均して $c$ ずつ同じ向きに動く。たとえば $c = 0.1$ なら、$\varepsilon_t$ による上下とは別に1期あたり平均0.1ずつ上へ動き、100期では平均10上がる。このように毎期一定の向きに進む動きをドリフトといい、このモデルをドリフトのあるランダムウォークという。この回で単に「ランダムウォーク」と書くときは、ドリフトのないものを指す。


## ランダムウォークの散らばり

最初の値を $y_0 = 0$ とすると、ランダムウォークの $t$ 期の値は、$\varepsilon_1$ から $\varepsilon_t$ までを順に足し合わせた合計になる。`.cumsum()` は、列の値を先頭から順に足し合わせた合計（累積和）を求める関数なので、ランダムウォークは乱数の列に `.cumsum()` を付ければ作れる。同じ式で5つの系列を250期分作ろう。


In [ ]:
rng = np.random.default_rng(1)

fig, ax = plt.subplots(figsize=(9, 4))
for k in range(5):
    walk = rng.normal(size=250).cumsum()
    ax.plot(walk, linewidth=0.8)
ax.axhline(0, color="gray", linewidth=0.8)
ax.set(title="ドリフトのないランダムウォーク（φ = 1、c = 0）の5つの系列", xlabel="t", ylabel="y")
format_axis(ax, dates=False)
fig.tight_layout()
plt.show()


5つの系列は同じ式から作ったが、t が200を過ぎてから上がり続けた系列もあれば、下がり続けた系列もある。ランダムウォークでは、見かけの上昇や下降が長く続いたり、向きが急に変わったりする。

t が大きくなるほど、5つの系列の開きは大きくなっている。t 期の値は $\varepsilon_1$ から $\varepsilon_t$ までの t 個の乱数の合計なので、その散らばり（標準偏差）は1期分の $\sqrt{t}$ 倍になる。第6回では、ドル・円の対数変化率を n 日分足し合わせた値の標準偏差が、1日の標準偏差の $\sqrt{n}$ 倍にほぼ等しいことを見た（21日分で2.81%、1日の0.62%の $\sqrt{21}$ 倍で2.84%）。対数で見たドル・円レートの散らばりも、ランダムウォークと同じように広がっている。


## 単位根

$\phi = 1$ の AR(1) モデルの系列には、$c$ の値によらず単位根があるという。ドリフトのないランダムウォークも、ドリフトのあるランダムウォークも、単位根がある系列である。単位根があると何が起こるのかを、ある期に入った $\varepsilon_t$ の影響がその後どれだけ残るかで調べよう。

$c = 0$ で最初の値を0とし、1期目だけ $\varepsilon_1 = 1$、2期目からは $\varepsilon_t = 0$ とする。AR(1) モデルの式に順に入れると $y_1 = 1$、$y_2 = \phi$、$y_3 = \phi^2$ となり、$\varepsilon_1$ が入ってから $k$ 期後の値は $\phi$ の $k$ 乗になる。この回の最初の解説で、予想と平均の差が1か月ごとに $\phi$ 倍になったのと同じ計算である。$\phi = 0.5$、$0.9$、$1.0$ の3つで30期後まで計算しよう。


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for phi in [0.5, 0.9, 1.0]:
    y = np.zeros(31)
    y[0] = 1
    for k in range(1, 31):
        y[k] = phi * y[k - 1]
    ax.plot(y, marker="o", markersize=3, label=f"φ = {phi}")
    print(f"φ = {phi}：5期後 {y[5]:.3f}、10期後 {y[10]:.3f}、30期後 {y[30]:.3f}")
ax.set(title="ある期に入った ε の影響（最初の大きさは1）", xlabel="何期後か", ylabel="残っている大きさ")
ax.legend()
format_axis(ax, dates=False)
fig.tight_layout()
plt.show()


$\phi = 0.5$ では、影響は5期後に0.031まで小さくなる。$\phi = 0.9$ では小さくなる速さが遅いが、30期後には0.042になる。$\phi = 1.0$ では影響は1のままで、何期たっても小さくならない。単位根があるとは、ある期に入った $\varepsilon_t$ の影響が消えずに残り続けるということである。ランダムウォークの値が $\varepsilon_1$ から $\varepsilon_t$ までの合計になるのは、どの期の $\varepsilon_t$ も同じ大きさのまま残るからである。

ドル・円レートでいえば、第5回で説明した雇用統計の数字のような新しい情報でレートが1円動いたとき、その1円の動きが後の日まで残り続けることにあたる。

「単位根」という名前は、AR(1) モデルの傾き $\phi$ から作る方程式 $1 - \phi z = 0$ からきている。この方程式の解は $z = 1 / \phi$ で、$-1 < \phi < 1$ なら解の絶対値は1より大きく、$\phi = 1$ なら解はちょうど1になる。数学では方程式の解のことを「根（こん）」ともいい、大きさが1であることを「単位」という。$\phi = 1$ のときの解1を「単位根」と呼ぶのはこのためである。


### 定常な系列

平均や散らばりが時間によって変わらない系列を、定常な系列という。$-1 < \phi < 1$ の AR(1) モデルは定常で、$\varepsilon_t$ の影響は $\phi$ 倍ずつ小さくなり、平均から離れても平均へ戻っていく。単位根がある系列は定常でない。「ランダムウォークの散らばり」の節で見たように、$t$ が大きくなるほど散らばりが大きくなるからである。

単位根がある系列でも、前の期からの変化を取ると定常な系列になる。$y_t = y_{t-1} + \varepsilon_t$ の両辺から $y_{t-1}$ を引くと $y_t - y_{t-1} = \varepsilon_t$ となり、前の期からの変化は乱数 $\varepsilon_t$ そのものである。ドリフトがあるときは $y_t - y_{t-1} = c + \varepsilon_t$ で、変化は平均 $c$ のまわりを上下する。どちらの変化も、平均と散らばりは時間によって変わらない。

乱数で作った系列で確かめよう。$\phi = 0.5$ の AR(1) モデルの系列、ランダムウォーク、そのランダムウォークの前の期からの変化を1000期分作り、平均と標準偏差を前半と後半で比べる。`.iloc[:500]` は先頭から500個を、`.iloc[500:]` は501個目から最後までを取り出す。


In [ ]:
rng = np.random.default_rng(7)
eps = rng.normal(size=1000)

ar = np.zeros(1000)
for t in range(1, 1000):
    ar[t] = 0.5 * ar[t - 1] + eps[t]
walk = pd.Series(eps.cumsum())

names = ["φ = 0.5 の AR(1) モデルの系列", "ランダムウォーク", "ランダムウォークの前の期からの変化"]
series_list = [pd.Series(ar), walk, walk.diff().dropna()]

fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=True)
for k in range(3):
    y = series_list[k]
    axes[k].plot(y, linewidth=0.6)
    axes[k].axhline(0, color="gray", linewidth=0.8)
    axes[k].axvline(500, color="gray", linewidth=0.8)
    axes[k].set(title=names[k], ylabel="y")
    format_axis(axes[k], dates=False)
    first = y.iloc[:500]
    second = y.iloc[500:]
    print(f"{names[k]}：平均 前半 {first.mean():.2f}、後半 {second.mean():.2f}／標準偏差 前半 {first.std():.2f}、後半 {second.std():.2f}")
axes[2].set_xlabel("t")
fig.tight_layout()
plt.show()


図の縦の線が前半と後半の境目である。$\phi = 0.5$ の系列は、平均が前半 −0.25、後半 −0.03、標準偏差が1.06と1.14で、前半と後半でほとんど変わらない。ランダムウォークは平均が −32.15 と −79.44、標準偏差が13.94と6.25で、前半と後半で大きく違い、0へ戻らずに離れていく。同じランダムウォークの前の期からの変化は、平均が −0.13 と −0.02、標準偏差が0.94と0.94で、前半と後半でほとんど変わらない。定常でないランダムウォークも、前の期からの変化を取ると、平均と散らばりが時間によって変わらない系列になった。


## ドル・円レートの AR(1) モデル

東京17時のドル・円レートに AR(1) モデルを当てはめる。第7回と同じく、前日の値と当日の値を同じ行に並べた表 `levels` を作り、2000〜2023年の値で切片と傾きを求める。


In [ ]:
levels = pd.DataFrame({
    "前日の値": rate.shift(1),
    "当日の値": rate,
}).dropna()

train_level = levels.loc[:"2023"]
model_level = sm.OLS(train_level["当日の値"], sm.add_constant(train_level["前日の値"])).fit()
c_level = model_level.params["const"]
phi_level = model_level.params["前日の値"]
print(f"切片 c：{c_level:.3f}、傾き φ：{phi_level:.4f}（{len(train_level)} 日）")
print(f"1 − 5.3 ÷ 日数：{1 - 5.3 / len(train_level):.4f}")


傾きは0.9994で、1にきわめて近い。1より少し小さいことから、レートはいずれ平均へ戻ると考えたくなる。しかし $\phi$ が本当に1でも、最小二乗法で求めた傾きは1より少し小さく出やすく、その目安は観測の数を T として 1 − 5.3 ÷ T になる。2000〜2023年の5883日では0.9991で、求めた傾き0.9994はこの目安とほぼ同じである。ドル・円レートの傾きは、$\phi = 1$ と見分けがつかない。

$\phi$ が1に近い系列では、過去の平均は予想の役に立たないはずである。2024年の各日のレートを、2000〜2023年の平均で予想した場合と、今日の値で予想した場合の誤差を比べよう。


In [ ]:
past_mean = rate.loc[:"2023"].mean()
test_level = levels.loc["2024"]
error_mean = (test_level["当日の値"] - past_mean).abs().mean()
error_naive = (test_level["当日の値"] - test_level["前日の値"]).abs().mean()
print(f"2000〜2023年の平均：{past_mean:.1f} 円／ドル")
print(f"平均による予想の誤差の大きさの平均（2024年）：{error_mean:.2f} 円／ドル")
print(f"今日の値による予想の誤差の大きさの平均（2024年）：{error_naive:.2f} 円／ドル")


2000〜2023年の平均は109.2円で、この値を2024年の予想にすると、誤差の大きさの平均は42.42円になった。今日の値をそのまま使う予想の0.70円の60倍を超える。2024年のレートは平均の109.2円から遠く離れたまま動き、平均へ戻る動きは見られなかった。$\phi$ が1に近い系列では、過去の平均から離れた値がそのまま続くので、平均は予想の役に立たない。次の日の予想には今日の値を使うのがよい。

レートの値と前日からの変化を、第6回と同じように上下に並べた図で比べよう。上の図の灰色の横線は、2000〜2023年の平均である。


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
axes[0].plot(rate.index, rate, linewidth=0.8, label="ドル・円レート")
axes[0].axhline(past_mean, color="gray", linewidth=0.8, label="2000〜2023年の平均")
axes[0].set(title="ドル・円レート（東京17時）", ylabel="円／ドル")
axes[0].legend()
axes[1].plot(change.index, change, linewidth=0.5)
axes[1].axhline(0, color="gray", linewidth=0.8)
axes[1].set(title="前日からの変化（東京17時）", ylabel="円／ドル")
for ax in axes:
    format_axis(ax)
fig.tight_layout()
plt.show()

print(f"前日からの変化の平均（2000〜2025年）：{change.mean():.3f} 円／ドル")
print(f"前日からの変化の大きさの平均（2000〜2025年）：{change.abs().mean():.3f} 円／ドル")


上の図のレートの値は、平均の線の片側に長く離れている。2009〜2013年は平均より下に、2022〜2025年は平均より上にあった。下の図の前日からの変化は、どの時期でも0のまわりを上下している。2024年8月の大きな円高のように0から大きく離れる日があっても、離れたままの時期は続かない。前日からの変化の平均は0.008円で、変化の大きさの平均0.503円の2%ほどである。ドリフトがあるとしても、1日あたりの大きさはごく小さい。

図の動きは、AR(1) モデルの傾きと合っている。レートの値の傾きは0.9994で単位根がある系列と見分けがつかず、平均へ戻る力が見えない。第7回で前日の変化に当てはめた傾きは −0.0142 とほぼ0で、前日の変化は定常な系列のように0のまわりを上下する。第5〜7回でレートの値の代わりに変化や対数変化率を分析に使ってきたのは、このためでもある。


## 演習1：単位根とレートの予想

1. 問1の計算欄のコードは、FRBのニューヨーク正午のユーロ・ドル（ドル／ユーロ）のレートの値に AR(1) モデルを当てはめ、傾きと 1 − 5.3 ÷ 日数の値を表示する。2024年の各日のレートを2000〜2023年の平均で予想した場合と、今日の値で予想した場合の誤差の大きさの平均も表示する。セルを実行し、4つの値を答えよ。また、ユーロ・ドルのレートの予想に過去の平均が役に立つかを答えよ。
2. 次の(a)〜(c)は正しいか。正しいか誤りかを答え、それぞれ理由を1文で書け。
   - (a) ドル・円レートに当てはめた AR(1) モデルの傾きは0.9994で1より小さいので、レートはいずれ2000〜2023年の平均の109.2円へ戻っていく。
   - (b) ドリフトのないランダムウォークでは、期間が長くなるほど値の散らばりが大きくなる。
   - (c) ドリフトのないランダムウォークで上昇が長く続いたあとは、次の期も上がると予想できる。
3. Cさんは「ドル・円レートの2000〜2023年の平均は109.2円だった。今は150円台なので、いずれ109円台へ戻るはずだ。今のうちにドルを売って円にしておけば得をする」と考えた。この回の結果を使って、Cさんの考えの問題点を3〜5文で説明せよ。

追加問題：本文で求めたドル・円レートの AR(1) モデル（切片 `c_level`、傾き `phi_level`）で2024年の各日のレートを予想し、誤差の大きさの平均を求めよ。今日の値をそのまま使う予想の誤差の大きさの平均と比べ、どちらの予想がよいかを答えよ。


In [ ]:
# 問1の計算欄：ユーロ・ドル（FRB）のレートの値の AR(1) モデルと予想の誤差
eur_usd = fed["eur_usd"].dropna()
eur_levels = pd.DataFrame({
    "前日の値": eur_usd.shift(1),
    "当日の値": eur_usd,
}).dropna()

eur_level_train = eur_levels.loc[:"2023"]
model_eur_level = sm.OLS(eur_level_train["当日の値"], sm.add_constant(eur_level_train["前日の値"])).fit()
print(f"傾き φ：{model_eur_level.params['前日の値']:.4f}（{len(eur_level_train)} 日）")
print(f"1 − 5.3 ÷ 日数：{1 - 5.3 / len(eur_level_train):.4f}")

eur_past_mean = eur_usd.loc[:"2023"].mean()
eur_test = eur_levels.loc["2024"]
eur_error_mean = (eur_test["当日の値"] - eur_past_mean).abs().mean()
eur_error_naive = (eur_test["当日の値"] - eur_test["前日の値"]).abs().mean()
print(f"平均による予想の誤差の大きさの平均（2024年）：{eur_error_mean:.4f} ドル／ユーロ")
print(f"今日の値による予想の誤差の大きさの平均（2024年）：{eur_error_naive:.4f} ドル／ユーロ")


In [ ]:
# 追加問題の計算欄
# 2024年の前日の値は test_level["前日の値"]、当日の値は test_level["当日の値"] で取り出せる。


### 演習1の記入欄

- 問1：傾き、1 − 5.3 ÷ 日数の値、2つの予想の誤差の大きさの平均、過去の平均が予想の役に立つか：
- 問2(a)：正しいか誤りか、その理由：
- 問2(b)：正しいか誤りか、その理由：
- 問2(c)：正しいか誤りか、その理由：
- 問3（3〜5文）：
- 追加問題：AR(1) モデルの予想の誤差の大きさの平均、今日の値による予想との比較：


<details>
<summary>演習1のヒント</summary>

問1では、2つの予想の誤差の大きさの平均を比べよう。

問2の(a)は「ドル・円レートの AR(1) モデル」の節で、傾きと 1 − 5.3 ÷ 日数の値を比べよう。(b)は「ランダムウォークの散らばり」の節の図を見よう。(c)は式 $y_t = y_{t-1} + \varepsilon_t$ で、次の期の値が今の値からどう決まるかを考えよう。

問3では、ドル・円レートの AR(1) モデルの傾きと、2024年の平均による予想の誤差の大きさの平均を使おう。

追加問題の予想は、第7回で月ごとの変動の大きさを予想したときと同じく、`c_level + phi_level * 前日の値` で計算できる。

</details>


## 出典

[日本銀行「外国為替市況」](https://www.boj.or.jp/statistics/market/forex/fxdaily/index.htm)の東京市場ドル・円スポット、17時時点（円／ドル）を使用。配布データの対象期間は2000〜2025年である。

第7回 演習1の解説と演習1の問1で使うユーロ・ドルは [FRB H.10（為替）](https://www.federalreserve.gov/releases/h10/)による。
